# 09 — Convergence à l'échelle : l'encodage décide de la tractabilité

**Clôture du bloc meal-planner (Epic #4677).** La modélisation déclarative ([06](06_Meal_Planner_Modelisation.ipynb)) a posé les modèles du planificateur sur un corpus jouet ; la couche de données ([07](07_Meal_Planner_Data_External.ipynb)) a construit le corpus réel — recettes RecipeML appariées à Ciqual ANSES 2025 **sans faux positifs** et agrégées **pondérées par la masse**. Ce notebook branche enfin le solveur sur ces données réelles, et y rencontre le problème que les corpus jouets masquaient : à l'échelle, l'encodage naïf explose dès la **construction**, l'encodage le plus **compact** (théorie des tableaux) devient **insoluble**, et seul l'encodage **one-hot pseudo-booléen** se construit instantanément et se résout.

> **Stack : B (API `Microsoft.Z3` brute)** — le cœur de la leçon est l'encodage **pseudo-booléen** (`MkPBEq` / `MkPBLe` / `MkPBGe`), une famille de contraintes que le DSL Z3.Linq n'expose pas (cf. [README, « Les deux stacks »](README.md#les-deux-stacks)).

> **Question de convergence.** Le problème fidèle reste-t-il *tractable* quand on remonte à l'échelle réelle (7 menus × 5 créneaux × ~2 200 recettes × constituants Ciqual) ? La puissance brute du solveur Z3 suffit-elle, ou le choix d'**encodage** décide-t-il seul de la tractabilité ?
>
> **Insight clé (formulation du problème).** Plus de recettes = plus de **solutions possibles**, pas plus de contraintes : des recettes supplémentaires sont des **contraintes enablantes** (elles élargissent l'espace des modèles). C'est l'encodage — pas le solveur — qui décide si cet espace est explorable.


## Objectifs d'apprentissage

- Consommer la **couche de données** construite par le notebook 07 (cache JSON : vecteurs nutritionnels **pondérés par la masse**, appariement curé sans faux positifs).
- Comprendre pourquoi un encodage SMT **naïf** explose dès la **construction** à l'échelle.
- Découvrir que la **compacité d'écriture** (théorie des tableaux) n'implique pas la **résolubilité**.
- Maîtriser l'encodage **one-hot pseudo-booléen** (`MkPBEq` / `MkPBLe` / `MkPBGe`) qui passe à l'échelle.

## Prérequis

- Notebooks [06](06_Meal_Planner_Modelisation.ipynb) (modélisation déclarative du menu) et [07](07_Meal_Planner_Data_External.ipynb) (couche de données Ciqual × RecipeML).
- Notebooks [04](04_Array_Theory.ipynb) (théorie des tableaux Z3) et [14](14_Optimize_MaxSAT.ipynb) (`Optimize` / pseudo-booléen).


## 1. Données : le cache solveur-usable du notebook 07

Ce notebook ne refait **aucun** data-engineering : il charge `data/meals/mealplan_cache.json`, le sous-ensemble **solveur-usable** (couverture d'appariement ≥ 80 %) sérialisé par le [notebook 07](07_Meal_Planner_Data_External.ipynb). Chaque recette y porte un vecteur nutritionnel (énergie kJ, protéines, glucides, lipides, sel) agrégé **pondéré par la masse réelle des ingrédients** — le contraire du raccourci per-100g qui rendait toute somme fictive.

Si le cache est absent, exécutez d'abord le notebook 07 de bout en bout (il télécharge au besoin ses sources via `python download_meal_data.py --areas Ciqual RecipeML`).


In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Microsoft.Z3;
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Text.Json;
using System.Diagnostics;

var CACHE = Path.Combine("data/meals", "mealplan_cache.json");
Console.WriteLine(File.Exists(CACHE)
    ? $"Cache present : {CACHE} ({new FileInfo(CACHE).Length / 1024} Ko)"
    : "Cache absent : executez d'abord le notebook 07 (couche de donnees) de bout en bout.");


Cache present : data/meals\mealplan_cache.json (270 Ko)


In [2]:
// ---- chargement du cache solveur-usable produit par le notebook 07 ----
var sw = Stopwatch.StartNew();
var doc = JsonDocument.Parse(File.ReadAllText(CACHE));
var root = doc.RootElement;
var constituants = root.GetProperty("constituants").EnumerateArray().Select(x => x.GetString()).ToArray();
int C = constituants.Length;
var plats = new List<(string title, decimal[] vec, List<string> cats)>();
foreach (var r in root.GetProperty("recipes").EnumerateArray())
{
    var t = r.GetProperty("title").GetString().Trim();
    plats.Add((t.Length > 40 ? t.Substring(0, 40) : t,
               r.GetProperty("vec").EnumerateArray().Select(x => x.GetDecimal()).ToArray(),
               r.GetProperty("cats").EnumerateArray().Select(x => x.GetString()).ToList()));
}
int R = plats.Count;
sw.Stop();
Console.WriteLine($"Cache charge en {sw.Elapsed.TotalSeconds:F1}s : R={R} recettes solveur-usables (sur {root.GetProperty("n_total").GetInt32()} brutes), C={C} constituants.");
Console.WriteLine("Quartiles par constituant (echelle : recette ENTIERE, agregation ponderee par la masse) :");
for (int c = 0; c < C; c++)
{
    var vals = plats.Select(p => p.vec[c]).OrderBy(v => v).ToList();
    decimal P(double q) => vals[(int)(q * (vals.Count - 1))];
    var nom = constituants[c].Length > 38 ? constituants[c].Substring(0, 38) : constituants[c];
    Console.WriteLine($"   [{c}] {nom,-38} P25={Math.Round(P(0.25), 1),8}  mediane={Math.Round(P(0.5), 1),8}  P75={Math.Round(P(0.75), 1),8}");
}


Cache charge en 0,0s : R=2387 recettes solveur-usables (sur 5424 brutes), C=5 constituants.


Quartiles par constituant (echelle : recette ENTIERE, agregation ponderee par la masse) :


   [0] Energie, Règlement UE N° 1169/2011 (kJ P25=  3253,1  mediane=  6818,1  P75= 12163,9


   [1] Protéines, N x facteur de Jones (g/100 P25=    11,5  mediane=    34,4  P75=    71,4


   [2] Glucides (g/100 g)                     P25=    31,5  mediane=   122,9  P75=   328,9


   [3] Lipides (g/100 g)                      P25=    22,7  mediane=    72,3  P75=   162,6


   [4] Sel chlorure de sodium (g/100 g)       P25=     1,2  mediane=     3,8  P75=     8,3


### Interprétation : ce que la couche 07 garantit (et ce qu'elle ne garantit pas)

Le cache livre des recettes **déjà appariées** (matcheur curé du notebook 07 : `White sugar` → *Sugar, white*, et non *White pudding* — zéro faux positif toléré) et **déjà pesées** (quantités culinaires converties en grammes, agrégation pondérée par la masse). Les premières ébauches de ce notebook refaisaient ici un appariement naïf par sac-de-mots — avec ses faux positifs — et sommaient les teneurs per-100g comme si chaque ingrédient pesait 100 g : c'est exactement le travail que le notebook 07 a assaini, et qu'on ne duplique plus.

Les quartiles affichés ci-dessus servent à **calibrer les bornes du théorème** (section suivante) : les vecteurs étant désormais à l'échelle de la **recette entière** (pas de la portion — RecipeML porte un `<yield>` que la couche 07 n'exploite pas encore), les fenêtres nutritionnelles par menu doivent être posées depuis ces ordres de grandeur **mesurés**, pas depuis des constantes héritées d'un autre encodage des données.


In [3]:
// ---- parametres du theoreme partages par les trois encodages ----
int NMENUS = 7, NPLATS = 5;
// valeurs entieres par constituant (kJ, g), a l'echelle de la RECETTE ENTIERE (agregation ponderee, cf. 07).
var vint = new int[C][];
for (int c = 0; c < C; c++) vint[c] = plats.Select(p => (int)Math.Round(p.vec[c])).ToArray();
// restrictions patient CALIBREES sur les quartiles mesures ci-dessus : les vecteurs etant a l'echelle de la
// recette entiere (et non per-100g), des constantes heritees d'un autre encodage n'auraient aucun sens.
decimal Pq(int c, double q) { var v = plats.Select(p => p.vec[c]).OrderBy(x => x).ToList(); return v[(int)(q * (v.Count - 1))]; }
int loE = NPLATS * (int)Pq(0, 0.20), hiE = NPLATS * (int)Pq(0, 0.80);   // energie/menu : bande large autour de l'IQR
int loP = NPLATS * (int)Pq(1, 0.30);                                    // proteines/menu : plancher realiste
int hiS = Math.Max(1, NPLATS * (int)Pq(4, 0.70));                       // sel/menu : plafond contraignant
var restr = new (int c, int lo, int hi)[] { (0, loE, hiE), (1, loP, -1), (4, -1, hiS) };
Console.WriteLine($"Theoreme : {NMENUS} menus x {NPLATS} plats, sur R={R} recettes (vecteurs recette entiere).");
Console.WriteLine($"   energie/menu in [{loE},{hiE}] kJ, proteines/menu >= {loP} g, sel/menu <= {hiS} g.");


Theoreme : 7 menus x 5 plats, sur R=2387 recettes (vecteurs recette entiere).


   energie/menu in [13230,71605] kJ, proteines/menu >= 70 g, sel/menu <= 35 g.


## 3. Trois encodages du meme theoreme, trois comportements

Le theoreme est fixe : **7 menus x 5 plats**, chaque plat **distinct** sur la semaine (variete), chaque
menu dans une **fenetre nutritionnelle**. Seul l'**encodage** SMT change. On va voir trois comportements
radicalement differents sur le **meme** corpus reel.

### 3.1 Encodage naif (disjonction) -- explose des la construction

L'encodage naif (celui du `Create` original) introduit une variable de nutrition par creneau et, pour
**chaque recette**, la disjonction `creneau != r OU nutrition == ligne[r]`. Le nombre d'assertions croit
en `menus x plats x R x constituants` : on ne mesure meme pas la resolution, juste la **construction**.

In [4]:
// ---- naif : sonde de temps de CONSTRUCTION (la resolution ne demarre meme pas a l'echelle) ----
long BuildNaive(int rcap)
{
    using var c = new Context();
    var so = c.MkSolver();
    var pid = new IntExpr[NMENUS][];
    for (int m = 0; m < NMENUS; m++) pid[m] = Enumerable.Range(0, NPLATS).Select(p => (IntExpr)c.MkIntConst($"q_{m}_{p}")).ToArray();
    long na = 0;
    var swp = Stopwatch.StartNew();
    for (int m = 0; m < NMENUS; m++)
        for (int p = 0; p < NPLATS; p++)
        {
            so.Assert(c.MkAnd(c.MkGe(pid[m][p], c.MkInt(0)), c.MkLt(pid[m][p], c.MkInt(rcap))));
            for (int rr = 0; rr < rcap; rr++)
                foreach (var (cc, lo, hi) in restr)
                {
                    var nutr = (IntExpr)c.MkIntConst($"n_{m}_{p}_{cc}");
                    so.Assert(c.MkOr(c.MkNot(c.MkEq(pid[m][p], c.MkInt(rr))), c.MkEq(nutr, c.MkInt(vint[cc][rr]))));
                    na++;
                }
        }
    swp.Stop();
    Console.WriteLine($"  naif R={rcap,5} : {na,8} disjonctions construites en {swp.Elapsed.TotalSeconds:F2}s (CONSTRUCTION seule, sans resolution)");
    return na;
}
foreach (var rcap in new[] { 100, 300, Math.Min(R, 1000) }) BuildNaive(rcap);
Console.WriteLine("  -> le cout de construction croit lineairement en R x menus x plats x contraintes :");
Console.WriteLine("     a l'echelle reelle, le solveur n'a meme pas commence a chercher une solution.");

  naif R=  100 :    10500 disjonctions construites en 0,11s (CONSTRUCTION seule, sans resolution)


  naif R=  300 :    31500 disjonctions construites en 0,24s (CONSTRUCTION seule, sans resolution)


  naif R= 1000 :   105000 disjonctions construites en 0,94s (CONSTRUCTION seule, sans resolution)


  -> le cout de construction croit lineairement en R x menus x plats x contraintes :


     a l'echelle reelle, le solveur n'a meme pas commence a chercher une solution.


### 3.2 Theorie des tableaux -- compacte a ecrire, mais insoluble

L'encodage par **theorie des tableaux** (notebook 04) est **compact** : un `Array` par constituant
(chaine de `Store` sur les R valeurs), un index entier par creneau, et `Select(arr, index)` pour lire la
valeur. Le nombre de **contraintes** devient independant de R. Surprise : Z3 retourne **`unknown`** -- il
ne sait pas resoudre les longues chaines de `Store` **symboliques**. *Compacite d'ecriture != resolubilite.*

In [5]:
// ---- theorie des tableaux : compact, mais Z3 -> unknown sur de longues chaines de Store symboliques ----
{
    using var c = new Context();
    var so = c.MkSolver();
    so.Set("timeout", (uint)15000);                                 // plafond 15s : on attend `unknown`
    var arrs = new ArrayExpr[restr.Length];
    for (int ci = 0; ci < restr.Length; ci++)
    {
        var (cc, lo, hi) = restr[ci];
        var a = c.MkConstArray(c.IntSort, c.MkInt(0));
        for (int r = 0; r < R; r++) a = c.MkStore(a, c.MkInt(r), c.MkInt(vint[cc][r]));    // chaine de R Store
        arrs[ci] = a;
    }
    var pid = new IntExpr[NMENUS][];
    for (int m = 0; m < NMENUS; m++) pid[m] = Enumerable.Range(0, NPLATS).Select(p => (IntExpr)c.MkIntConst($"a_{m}_{p}")).ToArray();
    var flat = (from m in Enumerable.Range(0, NMENUS) from p in Enumerable.Range(0, NPLATS) select pid[m][p]).ToArray();
    foreach (var v in flat) { so.Assert(c.MkGe(v, c.MkInt(0))); so.Assert(c.MkLt(v, c.MkInt(R))); }
    so.Assert(c.MkDistinct(flat));                                   // variete : indices distincts
    for (int m = 0; m < NMENUS; m++)
        for (int ci = 0; ci < restr.Length; ci++)
        {
            var (cc, lo, hi) = restr[ci];
            var tot = c.MkAdd(Enumerable.Range(0, NPLATS).Select(p => (ArithExpr)c.MkSelect(arrs[ci], pid[m][p])).ToArray());
            if (lo >= 0) so.Assert(c.MkGe(tot, c.MkInt(lo)));
            if (hi >= 0) so.Assert(c.MkLe(tot, c.MkInt(hi)));
        }
    var sw2 = Stopwatch.StartNew(); var rr = so.Check(); sw2.Stop();
    Console.WriteLine($"theorie des tableaux ({NMENUS * NPLATS} index, chaines de Store de longueur {R}) : {rr} en {sw2.Elapsed.TotalSeconds:F1}s");
    Console.WriteLine("  -> compact a ECRIRE, mais insoluble : Z3 ne tranche pas les Store symboliques empiles. Compacite != resolubilite.");
}

theorie des tableaux (35 index, chaines de Store de longueur 2387) : UNKNOWN en 15,1s


  -> compact a ECRIRE, mais insoluble : Z3 ne tranche pas les Store symboliques empiles. Compacite != resolubilite.


### 3.3 One-hot pseudo-booleen -- l'encodage qui passe a l'echelle

L'encodage **one-hot** introduit un booleen `sel[m][p][r]` ("la recette r occupe le creneau p du menu m").
Trois familles de contraintes, toutes **pseudo-booleennes natives** de Z3 :

- `MkPBEq(.,1)` par creneau : **exactement une** recette par creneau.
- `MkPBLe(.,1)` par recette sur toute la semaine : chaque recette **au plus une fois** (= variete / `Distinct`).
- `MkPBGe` / `MkPBLe` **ponderes** par menu et par constituant : la fenetre nutritionnelle devient une
  **somme ponderee de booleens** (coefficient = valeur de la recette), traitee par le **solveur
  pseudo-booleen** de Z3 -- pas par l'arithmetique lineaire generale.

C'est cet encodage qui passe a l'echelle : les recettes sont des **contraintes enablantes**, le solveur
trouve un modele sans enumerer. La **construction** est quasi instantanee meme a R~1000.

In [6]:
// ---- one-hot pseudo-booleen : exactly-one + variete + bandes ponderees (MkPBGe / MkPBLe) ----
var ctx = new Context();
var s = ctx.MkSolver();
var swB = Stopwatch.StartNew();
var sel = new BoolExpr[NMENUS][][];
for (int m = 0; m < NMENUS; m++) { sel[m] = new BoolExpr[NPLATS][]; for (int p = 0; p < NPLATS; p++) sel[m][p] = Enumerable.Range(0, R).Select(r => ctx.MkBoolConst($"s_{m}_{p}_{r}")).ToArray(); }
var onesR = Enumerable.Repeat(1, R).ToArray();
for (int m = 0; m < NMENUS; m++) for (int p = 0; p < NPLATS; p++) s.Assert(ctx.MkPBEq(onesR, sel[m][p], 1));      // exactement 1 recette / creneau
var onesMP = Enumerable.Repeat(1, NMENUS * NPLATS).ToArray();
for (int r = 0; r < R; r++) { var col = (from m in Enumerable.Range(0, NMENUS) from p in Enumerable.Range(0, NPLATS) select sel[m][p][r]).ToArray(); s.Assert(ctx.MkPBLe(onesMP, col, 1)); }  // variete : <= 1x / semaine
// fenetre nutritionnelle = somme ponderee de booleens (PB native), coeff = valeur de la recette
for (int m = 0; m < NMENUS; m++) foreach (var (c, lo, hi) in restr)
{
    var bools = (from p in Enumerable.Range(0, NPLATS) from r in Enumerable.Range(0, R) select sel[m][p][r]).ToArray();
    var coeffs = (from p in Enumerable.Range(0, NPLATS) from r in Enumerable.Range(0, R) select vint[c][r]).ToArray();
    if (hi >= 0) s.Assert(ctx.MkPBLe(coeffs, bools, hi));
    if (lo >= 0) s.Assert(ctx.MkPBGe(coeffs, bools, lo));
}
swB.Stop();
var swS = Stopwatch.StartNew(); var res = s.Check(); swS.Stop();
Console.WriteLine($"one-hot R={R} ({NMENUS * NPLATS * R} booleens) : construction {swB.Elapsed.TotalSeconds:F1}s, resolution {swS.Elapsed.TotalSeconds:F1}s -> {res}");
if (res == Status.SATISFIABLE)
{
    var mo = s.Model;
    for (int m = 0; m < NMENUS; m++)
    {
        var names = new List<string>();
        for (int p = 0; p < NPLATS; p++) for (int r = 0; r < R; r++) if (mo.Eval(sel[m][p][r], true).IsTrue) names.Add(plats[r].title);
        Console.WriteLine($"  Menu {m + 1} : " + string.Join("  |  ", names.Select(n => n.Length > 18 ? n.Substring(0, 18) : n)));
    }
}

one-hot R=2387 (83545 booleens) : construction 0,9s, resolution 5,2s -> SATISFIABLE


  Menu 1 : 1930 Quick Tomato   |  12 Hour Salad  |  100% Crunch Bread  |  1-Pot: Creamy Chic  |  1-Pot Pastitsio Go


  Menu 2 : 1986 Winner Coconu  |  11 Minute Strawber  |  1-Pot: Mushroom an  |  1-Pot: Cheesy Turk  |  1-Pot Fuss-Free Ca


  Menu 3 : 1-Pot Mushroom and  |  1-Pot Creamy Chick  |  1-2-3 Sweet Dough  |  1-2-3 Meurbeteig D  |  1-2-3 Cookies


  Menu 4 : 10-Minute Lasagna  |  1-Pot Cheesy Turke  |  1-2-3-4 Cake with   |  1-2-3-4 Cake By Ja  |  1-2-3-4 Cake


  Menu 5 : 1-2-3-4-5 Cake  |  1-1-1 Cookies  |  1,000 Calorie-A-Bi  |  (Sort-Of) Sweet an  |  (Homemade Fresh) C


  Menu 6 : 1,2,3,4 Cake  |  ( From Bread Mix )  |  (Sort of Light) Bo  |  'sense and Sensibi  |  'ncapriata Di Fave


  Menu 7 : 21-Alarm Chile --L  |  'gimme Both' Pumpk  |  $100 Chocolate Cak  |  #1 Lemon Bars  |  #10 Cake


### Interpretation : la compacite ne fait pas la resolubilite

Trois encodages, un seul corpus :

| Encodage | Taille | Comportement a R~1000 |
|---|---|---|
| **Naif (disjonction)** | `menus x plats x R x C` assertions | **explose a la construction** |
| **Theorie des tableaux** | compact (R-independant) | **`unknown`** -- Store symboliques insolubles |
| **One-hot pseudo-booleen** | `menus x plats x R` booleens | **construction quasi-immediate + resolu** |

La lecon centrale : a l'echelle, **l'encodage prime sur la compacite**. L'encodage one-hot pseudo-booleen
repond a "plus de donnees" par "plus de modeles a trouver", pas "plus de cas a enumerer" -- c'est la
traduction concrete de "les recettes sont des contraintes enablantes".

> **Nuance (jusqu'au choix de la contrainte).** Meme **au sein** du one-hot, le detail compte : exprimer la
> fenetre nutritionnelle comme une **contrainte pseudo-booleenne native** (`MkPBGe` / `MkPBLe`, somme ponderee
> de booleens, traitee par le solveur PB) plutot que comme une **somme d'`ITE`** routee vers l'arithmetique
> lineaire generale change la resolution d'un facteur **~150x** (mesure lors d'une iteration precedente a ~1 000 recettes : ~0,7 s contre ~110 s).
> Le bon outil SMT pour une somme ponderee de booleens est le solveur pseudo-booleen, pas le solveur LIA.

## 4. Partitionnement par categorie : reduire R par creneau

Le one-hot plat traite les cinq creneaux de chaque menu de facon identique : chacun peut tirer dans les
**R** recettes. Or RecipeML porte `<categories><cat>` (Appetizers / Main dish / Desserts / ...). En
affectant **un pool par creneau** -- le creneau 1 ne tire que dans les entrees, le creneau 5 que dans les
desserts -- chaque creneau ne choisit plus que dans un **sous-ensemble** (R/creneau plus petit). Deux gains :

1. **Moins de booleens** : `sum_p |pool[p]| x menus` au lieu de `menus x plats x R` -- le solveur a
   structurellement moins de variables a poser.
2. **Fidelite a la contrainte d'ordre** du notebook 06 (entree -> plat -> dessert) : la structure du
   menu n'est plus emergente mais **imposee par construction**, et les menus produits sont coherents
   (une entree, un plat, un accompagnement, un pain, un dessert) au lieu de cinq desserts.

On reutilise le solveur **pseudo-booleen pondere** de la section 3.3, applique aux pools. C'est le pont
vers un planificateur a l'echelle du corpus complet (5 215 recettes brutes, ou le one-hot plat atteindrait
~183 000 booleens).

In [7]:
// ---- partitionnement par categorie : un pool de recettes par creneau ----
string[] COURSES = { "Entree", "Plat principal", "Accompagnement", "Pain", "Dessert" };
var COURSE_CATS = new[]
{
    new HashSet<string>(StringComparer.OrdinalIgnoreCase){ "Appetizers","Soups","Salads","Salad","Soup" },
    new HashSet<string>(StringComparer.OrdinalIgnoreCase){ "Main dish","Meats","Beef","Poultry","Seafood","Fish","Pasta","Casseroles","Chili","Pork","Chicken","Stews" },
    new HashSet<string>(StringComparer.OrdinalIgnoreCase){ "Vegetables","Vegetarian","Sauces","Sauce","Side dishes","Rice","Potatoes" },
    new HashSet<string>(StringComparer.OrdinalIgnoreCase){ "Breads","Bread","Muffins","Rolls","Biscuits" },
    new HashSet<string>(StringComparer.OrdinalIgnoreCase){ "Desserts","Cakes","Cake","Cookies","Chocolate","Fruits","Pies","Candy","Pastries" },
};
int CourseOf(List<string> cats)                                                     // 1er <cat> reconnu gagne ; defaut = plat principal
{
    foreach (var cat in cats) for (int k = 0; k < 5; k++) if (COURSE_CATS[k].Contains(cat)) return k;
    return 1;
}
var pool = new List<int>[5];
for (int k = 0; k < 5; k++) pool[k] = new List<int>();
for (int r = 0; r < R; r++) pool[CourseOf(plats[r].cats)].Add(r);                   // chaque recette -> exactement un pool
Console.WriteLine("Pools par creneau : " + string.Join(", ", Enumerable.Range(0, 5).Select(k => $"{COURSES[k]}={pool[k].Count}")));

var c3 = new Context();
var s3 = c3.MkSolver();
var swB3 = Stopwatch.StartNew();
// sel3[m][p][j] : dans le menu m, le creneau p prend la j-eme recette DE pool[p] -> |pool[p]| booleens / creneau
var sel3 = new BoolExpr[NMENUS][][];
long nb3 = 0;
for (int m = 0; m < NMENUS; m++)
{
    sel3[m] = new BoolExpr[NPLATS][];
    for (int p = 0; p < NPLATS; p++) { sel3[m][p] = Enumerable.Range(0, pool[p].Count).Select(j => c3.MkBoolConst($"c_{m}_{p}_{j}")).ToArray(); nb3 += pool[p].Count; }
}
for (int m = 0; m < NMENUS; m++) for (int p = 0; p < NPLATS; p++)                   // exactement 1 recette / creneau
    s3.Assert(c3.MkPBEq(Enumerable.Repeat(1, pool[p].Count).ToArray(), sel3[m][p], 1));
for (int p = 0; p < NPLATS; p++) for (int j = 0; j < pool[p].Count; j++)            // variete : chaque recette <= 1x / semaine
{
    var col = Enumerable.Range(0, NMENUS).Select(m => sel3[m][p][j]).ToArray();
    s3.Assert(c3.MkPBLe(Enumerable.Repeat(1, NMENUS).ToArray(), col, 1));
}
for (int m = 0; m < NMENUS; m++) foreach (var (c, lo, hi) in restr)                 // memes bandes PB ponderees qu'en 3.3
{
    var bools = (from p in Enumerable.Range(0, NPLATS) from j in Enumerable.Range(0, pool[p].Count) select sel3[m][p][j]).ToArray();
    var coeffs = (from p in Enumerable.Range(0, NPLATS) from j in Enumerable.Range(0, pool[p].Count) select vint[c][pool[p][j]]).ToArray();
    if (hi >= 0) s3.Assert(c3.MkPBLe(coeffs, bools, hi));
    if (lo >= 0) s3.Assert(c3.MkPBGe(coeffs, bools, lo));
}
swB3.Stop();
var swS3 = Stopwatch.StartNew(); var res3 = s3.Check(); swS3.Stop();
Console.WriteLine($"course-onehot : {nb3} booleens (contre {NMENUS * NPLATS * R} en one-hot plat) -- construction {swB3.Elapsed.TotalSeconds:F1}s, resolution {swS3.Elapsed.TotalSeconds:F1}s -> {res3}");
if (res3 == Status.SATISFIABLE)
{
    var mo = s3.Model;
    for (int m = 0; m < 3; m++)
    {
        var names = new List<string>();
        for (int p = 0; p < NPLATS; p++) for (int j = 0; j < pool[p].Count; j++) if (mo.Eval(sel3[m][p][j], true).IsTrue) names.Add($"{COURSES[p]} : {plats[pool[p][j]].title}");
        Console.WriteLine($"  Menu {m + 1} -> " + string.Join("  |  ", names.Select(n => n.Length > 30 ? n.Substring(0, 30) : n)));
    }
}

Pools par creneau : Entree=180, Plat principal=1355, Accompagnement=268, Pain=173, Dessert=411


course-onehot : 16709 booleens (contre 83545 en one-hot plat) -- construction 0,2s, resolution 0,9s -> SATISFIABLE


  Menu 1 -> Entree : 7-Up Salad  |  Plat principal : 1-Pot Cheesy   |  Accompagnement : Acapulco-Los   |  Pain : 100% Whole Wheat Bread   |  Dessert : 1-2-3-4-5 Cake


  Menu 2 -> Entree : 24-Hour Vegetable Sal  |  Plat principal : 1-2-3 Sweet D  |  Accompagnement : About Braisin  |  Pain : 100% Whole Wheat Bread   |  Dessert : 1,2,3,4 Cake


  Menu 3 -> Entree : 24-Hour Salad  |  Plat principal : 1-2-3-4 Cake   |  Accompagnement : Aaparagus wit  |  Pain : 100% Whole Wheat  |  Dessert : 1,000 Calorie-A-Bite


### Interpretation : la structure imposee aide le solveur

Le partitionnement divise le nombre de booleens (chaque creneau ne voit que son pool, pas tout le corpus)
**et** garantit des menus structures -- chaque ligne ci-dessus est bien *entree / plat / accompagnement /
pain / dessert*, la ou le one-hot plat de la section 3.3 pouvait empiler cinq desserts. La meme machinerie
pseudo-booleenne (`MkPBEq` / `MkPBLe` / `MkPBGe`) s'applique aux pools sans changement : c'est l'espace
d'indices qui retrecit, pas l'encodage. A l'echelle du corpus complet, on partitionnerait plus finement
(sous-categories, equilibrage des pools) -- mais le principe "un pool par creneau" suffit a casser la
croissance en `plats x R` en une croissance par pool.

## 5. Restriction patient : un menu vegetarien

L'utilisateur a observe que les recettes sont des **contraintes enablantes** : plus de recettes = plus de
solutions, le solveur converge plus facilement. Une **restriction patient** (regime, allergie, intolerance)
joue le role inverse -- elle *retrecit* l'espace des solutions sans en changer la taille d'encodage. On le
montre ici avec un **menu vegetarien** : il suffit d'interdire toute recette dont une `<cat>` RecipeML est une
categorie viande/poisson (`Meats`, `Beef`, `Poultry`, `Seafood`, `Fish`, `Pork`, `Chicken`, `Stews`), exactement
comme l'exclusion d'allergene -- une clause unaire `MkNot(sel...)` par booleen concerne. Aucune bande
nutritionnelle n'est touchee : l'encodage par pools de la section 4 est reutilise tel quel, on ajoute seulement
des contraintes negatives. Le corpus reste assez riche (pool *plat principal* a des centaines de recettes
non carnees : `Main dish`, `Pasta`, `Casseroles`...) pour que le probleme demeure SATISFIABLE.

In [8]:
// ---- restriction patient : menu vegetarien (exclusion categorielle sur les memes pools qu'en section 4) ----
var BANNED_VEG = new HashSet<string>(StringComparer.OrdinalIgnoreCase)
    { "Meats", "Beef", "Poultry", "Seafood", "Fish", "Pork", "Chicken", "Stews" };
int nForbidden = Enumerable.Range(0, R).Count(r => plats[r].cats.Any(cc => BANNED_VEG.Contains(cc)));

var c5 = new Context();
var s5 = c5.MkSolver();
var sel5 = new BoolExpr[NMENUS][][];                                                // meme forme que sel3 (pools)
for (int m = 0; m < NMENUS; m++)
{
    sel5[m] = new BoolExpr[NPLATS][];
    for (int p = 0; p < NPLATS; p++) sel5[m][p] = Enumerable.Range(0, pool[p].Count).Select(j => c5.MkBoolConst($"v_{m}_{p}_{j}")).ToArray();
}
for (int m = 0; m < NMENUS; m++) for (int p = 0; p < NPLATS; p++)                   // exactement 1 recette / creneau
    s5.Assert(c5.MkPBEq(Enumerable.Repeat(1, pool[p].Count).ToArray(), sel5[m][p], 1));
for (int p = 0; p < NPLATS; p++) for (int j = 0; j < pool[p].Count; j++)            // variete : chaque recette <= 1x / semaine
{
    var col = Enumerable.Range(0, NMENUS).Select(m => sel5[m][p][j]).ToArray();
    s5.Assert(c5.MkPBLe(Enumerable.Repeat(1, NMENUS).ToArray(), col, 1));
}
for (int m = 0; m < NMENUS; m++) foreach (var (c, lo, hi) in restr)                 // memes bandes nutritionnelles PB
{
    var bools = (from p in Enumerable.Range(0, NPLATS) from j in Enumerable.Range(0, pool[p].Count) select sel5[m][p][j]).ToArray();
    var coeffs = (from p in Enumerable.Range(0, NPLATS) from j in Enumerable.Range(0, pool[p].Count) select vint[c][pool[p][j]]).ToArray();
    if (hi >= 0) s5.Assert(c5.MkPBLe(coeffs, bools, hi));
    if (lo >= 0) s5.Assert(c5.MkPBGe(coeffs, bools, lo));
}
// LA restriction : interdire toute recette viande/poisson dans chaque creneau ou elle pourrait apparaitre
for (int m = 0; m < NMENUS; m++) for (int p = 0; p < NPLATS; p++) for (int j = 0; j < pool[p].Count; j++)
    if (plats[pool[p][j]].cats.Any(cc => BANNED_VEG.Contains(cc))) s5.Assert(c5.MkNot(sel5[m][p][j]));

var swS5 = Stopwatch.StartNew(); var res5 = s5.Check(); swS5.Stop();
Console.WriteLine($"vegetarien : {nForbidden} recettes viande/poisson interdites -- resolution {swS5.Elapsed.TotalSeconds:F1}s -> {res5}");
if (res5 == Status.SATISFIABLE)
{
    var mo5 = s5.Model;
    var names = new List<string>();
    for (int p = 0; p < NPLATS; p++) for (int j = 0; j < pool[p].Count; j++) if (mo5.Eval(sel5[0][p][j], true).IsTrue) names.Add($"{COURSES[p]} : {plats[pool[p][j]].title}");
    Console.WriteLine("  Menu vegetarien 1 -> " + string.Join("  |  ", names.Select(n => n.Length > 30 ? n.Substring(0, 30) : n)));
}

vegetarien : 316 recettes viande/poisson interdites -- resolution 0,8s -> SATISFIABLE


  Menu vegetarien 1 -> Entree : 4 B's Restaurant Toma  |  Plat principal : 1-Pot Creamy   |  Accompagnement : Acapulco Rice  |  Pain : 100% Whole Wheat Bread   |  Dessert : 1-2-3-4-5 Cake


## Exercices

### Exercice 1 — seuil de non-satisfiabilité
Resserrez la borne **haute d'énergie** par menu (constituant 0 ; la valeur calibrée `hiE` est affichée par la cellule des paramètres) jusqu'au seuil où le problème 7×5 devient `UNSATISFIABLE`. À partir de quelle énergie maximale cumulée le planificateur ne trouve-t-il plus de semaine valide ?

*Indice :* reprenez l'encodage one-hot pseudo-booléen de la section 3.3 en faisant varier le `hi` de la bande énergie dans `{ hiE, 3 * hiE / 4, hiE / 2, hiE / 4 }`, et observez le `Status` renvoyé par `s.Check()`.


In [9]:
// Exercice 1 : trouver le seuil d'energie maximale ou 7x5 devient UNSATISFIABLE.
// Indice : reprenez l'encodage one-hot de la section 3.3 ; faites varier la borne 'hi'
//          de la bande energie (constituant 0, ligne (0, loE, hiE) de restr) dans
//          { hiE, 3 * hiE / 4, hiE / 2, hiE / 4 } et affichez le Status de s.Check() pour chaque valeur.
// Etape 1 : encapsuler la section 3.3 dans une fonction SolveEnergyCap(int hi) -> Status.
// Etape 2 : boucler sur les seuils, reperer la bascule SAT -> UNSAT.
Console.WriteLine("Exercice 1 a completer : seuil UNSAT de la fenetre energetique.");


Exercice 1 a completer : seuil UNSAT de la fenetre energetique.


### Exercice 2 : plancher de glucides par menu
Ajoutez une **borne basse** sur les glucides (constituant d'index 2) de chaque menu, sur le modele one-hot
de la section 3.3.

In [10]:
// Exercice 2 : plancher de glucides par menu (constituant index 2).
// Indice : meme patron que la bande proteines. Avec l'encodage PB pondere :
//   var bools = (from p in Enumerable.Range(0,NPLATS) from r in Enumerable.Range(0,R) select sel[m][p][r]).ToArray();
//   var coeffs = (from p in Enumerable.Range(0,NPLATS) from r in Enumerable.Range(0,R) select vint[2][r]).ToArray();
//   s.Assert(ctx.MkPBGe(coeffs, bools, GLUCIDES_MIN));   // pour chaque menu m
// Etape 1 : choisir GLUCIDES_MIN (ex. 150).
// Etape 2 : ajouter la contrainte pour chaque menu, re-resoudre, verifier que chaque menu atteint le seuil.
Console.WriteLine("Exercice 2 a completer : plancher de glucides par menu.");

Exercice 2 a completer : plancher de glucides par menu.


### Exercice 3 : exclusion d'un allergene par mot-cle
Interdisez toute recette dont le titre contient un mot-cle (ex. `Peanut`, `Shrimp`).

In [11]:
// Exercice 3 : exclure les recettes dont le titre contient un mot-cle allergene.
// Indice :
//   string[] bannis = { "Peanut", "Shrimp", "Crab" };
//   for (int r = 0; r < R; r++)
//       if (bannis.Any(b => plats[r].title.IndexOf(b, StringComparison.OrdinalIgnoreCase) >= 0))
//           for (int m = 0; m < NMENUS; m++) for (int p = 0; p < NPLATS; p++) s.Assert(ctx.MkNot(sel[m][p][r]));
// Etape 1 : pre-calculer l'ensemble des indices interdits. Etape 2 : forcer sel = false. Etape 3 : re-resoudre.
Console.WriteLine("Exercice 3 a completer : exclusion d'allergene par mot-cle.");

Exercice 3 a completer : exclusion d'allergene par mot-cle.


### Exercice 4 : mesurer l'explosion
Faites varier R (100, 300, R complet) et comparez le temps de **construction** du naif au temps
**construction + resolution** du one-hot.

In [12]:
// Exercice 4 : tracer construction(naif) vs construction+resolution(one-hot) en fonction de R.
// Indice : reutilisez BuildNaive(rcap) de la section 3.1 ; pour le one-hot, encapsulez la section 3.3
//          dans une fonction SolveOneHot(int rcap) qui ne garde que les rcap premieres recettes.
// Etape 1 : pour rcap in {100, 300, R}, mesurer les deux temps.
// Etape 2 : observer que le naif croit en R (construction) tandis que le one-hot reste constructible.
Console.WriteLine("Exercice 4 a completer : mesure comparative de l'explosion.");

Exercice 4 a completer : mesure comparative de l'explosion.


## Conclusion

### Points clés à retenir

- La **séparation données / modélisation** paie : le notebook [07](07_Meal_Planner_Data_External.ipynb) livre un cache curé (appariement sans faux positifs, agrégation pondérée par la masse) ; ce notebook n'a plus qu'à encoder et résoudre. Un solveur ne rattrape jamais des données mal jointes.
- À l'échelle, **l'encodage prime sur la compacité** : one-hot pseudo-booléen (construit + résolu) > théorie des tableaux (compacte mais `unknown`) > naïf (explose à la construction).
- **« Plus de données = contraintes enablantes »** : bien encodé, un gros corpus se résout sans énumération.
- Le **partitionnement par catégorie** (section 4) réduit R par créneau **et** impose la structure du menu (entrée → dessert) : c'est le chemin vers le corpus RecipeML complet (5 215 recettes brutes).

### Clôture du bloc meal-planner (Epic #4677, capstone #4617)

Ce notebook clôt l'arc du planificateur de repas : [06](06_Meal_Planner_Modelisation.ipynb) pose les modèles Z3 sur un corpus jouet, [07](07_Meal_Planner_Data_External.ipynb) construit la couche de données réelle Ciqual × RecipeML, [08](08_Meal_Planner_Patient_Capstone.ipynb) pousse le théorème patient en capstone déclaratif, et ce notebook fait converger le problème fidèle à l'échelle réelle. La leçon structurante : la puissance du solveur Z3 ne compense **pas** un mauvais encodage — c'est la reformulation **one-hot pseudo-booléenne** qui fait converger le planificateur (les fenêtres nutritionnelles patient deviennent des contraintes cardinales pondérées natives `MkPBGe`/`MkPBLe`, et l'espace de recherche rétrécit aux sélecteurs booléens plutôt qu'à l'énumération disjonctive des compositions). La même leçon d'encodage vaut pour tout problème combinatoire où le corpus grossit.
